## Importing Libaries and Data

In [57]:
from sympy import *
import pandas as pd
from numpy import *
import matplotlib.pyplot as plt
from helpers import *
from implementations import *

In [ ]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("dataset")

In [77]:
y_train[y_train == -1] = 0  

df_x_train = pd.DataFrame(x_train, index=train_ids)
df_x_test = pd.DataFrame(x_test, index=test_ids)

df_y_train = pd.DataFrame(y_train, index=train_ids, columns=["HAS_DISEASE"])

## Explorarory Analysis

In [78]:
N = df_y_train.shape[0]
total_disease = (df_y_train["HAS_DISEASE"] == 1).sum()

print(f"Percentage of Patients with MICHD in the training set: {(total_disease/N)*100: .2f} %")

Percentage of Patients with MICHD in the training set:  8.83 %


In [ ]:
# Exploring missing values for each feature

fraction_missing = np.isnan(x_train).mean(axis=0)
keep = fraction_missing < 0.6 # we drop any columns with more than 60% missing values
x_train = x_train[:, keep]

median = np.nanmedian(x_train, axis=0)
x_train = np.where(np.isnan(x_train), median, x_train)

## Implementing Regularized Logistic Regression

In [115]:
# Splitting the training data into a training and a test set for hyperparameter tuning
def split_data(x, y, ratio, seed=1):
    """
    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.
    """
    # set seed
    np.random.seed(seed)
    N = x.shape[0]

    shuffled_indices = np.random.permutation(N)
    cutoff = int(np.floor(ratio * N))

    train_indices = shuffled_indices[:cutoff]
    test_indices = shuffled_indices[cutoff:]

    x_tr, x_te = x[train_indices], x[test_indices]
    y_tr, y_te = y[train_indices], y[test_indices]

    return x_tr, x_te, y_tr, y_te

x_tr, x_te, y_tr, y_te = split_data(x_train, y_train, 0.8, seed=1)

y_tr = y_tr.reshape(-1, 1)
y_te = y_te.reshape(-1, 1)

In [116]:
# Standardize from training data
mean = x_tr.mean(axis=0)
std = x_tr.std(axis=0)
std[std == 0] = 1

x_tr = (x_tr - mean) / std
x_te = (x_te - mean) / std

# Add the bias column to both
x_tr = np.c_[np.ones(x_tr.shape[0]), x_tr]
x_te = np.c_[np.ones(x_te.shape[0]), x_te]

In [122]:
initial_w = np.zeros((x_tr.shape[1], 1))
max_iters = 1000
lambda_ = 0

min_loss = np.inf
best_gamma = None

# Finding the best learning rate
for gamma in [0.5, 0.75, 1, 1.25, 1.5, 2]:
    w, loss = reg_logistic_regression(y_tr, x_tr, lambda_, initial_w, max_iters, gamma)
    if min_loss > loss:
        min_loss = loss
        best_gamma = gamma
    print(f"gamma = {gamma:<8} loss = {loss:.4f}")

print(f"Smallest Loss ={min_loss: .2f} Best gamma = {best_gamma}")

gamma = 0.5      loss = 0.2269


KeyboardInterrupt: 

In [ ]:
# Finding the best hyperparameter
lambdas = [0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

def log_loss(y, tx, w):
    z = tx @ w
    return np.mean(-y * z + np.logaddexp(0, z))

loss_tr, loss_te = [], []
for lam in lambdas:
    w, _ = reg_logistic_regression(y_tr, x_tr, lam, initial_w, max_iters, best_gamma)
    loss_tr.append(log_loss(y_tr, x_tr, w))
    loss_te.append(log_loss(y_te, x_te, w))

    print(f"lambda = {lam:<8} train = {loss_tr[-1]:.4f} test = {loss_te[-1]:.4f}")

best_lambda = lambdas[int(np.argmin(loss_te))]
print("best lambda:", best_lambda)

lambda = 0        train = 0.2274 test = 0.2247
lambda = 1e-06    train = 0.2274 test = 0.2247
lambda = 1e-05    train = 0.2274 test = 0.2247
lambda = 0.0001   train = 0.2275 test = 0.2248
lambda = 0.001    train = 0.2285 test = 0.2258
lambda = 0.01     train = 0.2519 test = 0.2497
lambda = 0.1      train = 0.5885 test = 0.5915
best lambda: 0


In [ ]:
#baseline log loss
p = y_tr.mean()
-(p*np.log(p) + (1-p)*np.log(1-p))

0.2997640727847556